In [12]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd

# Load the posts data
posts_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/cleaned/chunk_0_posts_cleaned.parquet"
posts_table = pq.read_table(posts_path)

print("Posts data schema:")
print(posts_table.schema)
print(f"Total posts: {len(posts_table)}")
print("\nFirst few rows of posts:")
print(posts_table.to_pandas().head())

# Load the profiles data
profiles_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/cleaned/profiles_cleaned.parquet"
profiles_table = pq.read_table(profiles_path)

print("\nProfiles data schema:")
print(profiles_table.schema)
print(f"Total profiles: {len(profiles_table)}")
print("\nFirst few rows of profiles:")
print(profiles_table.to_pandas().head())

Posts data schema:
did_id: int64
created_at: timestamp[us, tz=UTC]
Total posts: 5000000

First few rows of posts:
     did_id                       created_at
0  31955122 2023-09-16 19:36:58.558000+00:00
1  31955122 2024-11-15 16:22:35.542000+00:00
2  31955122 2024-11-16 21:00:29.703000+00:00
3  31955122 2024-11-22 20:30:39.146000+00:00
4  31955122 2024-12-13 18:34:28.434000+00:00

Profiles data schema:
did_id: int64
created_at: timestamp[us, tz=UTC]
joined_via_starter_pack: extension<arrow.json>
Total profiles: 32170299

First few rows of profiles:
   did_id                       created_at joined_via_starter_pack
0       1 2024-11-15 07:04:16.352000+00:00                    None
1       2 2024-11-27 18:21:32.298000+00:00                    None
2       3 2024-11-16 13:04:51.383000+00:00                    None
3       4 2024-11-27 14:06:44.280000+00:00                    None
4       5 2024-12-08 15:24:03.663000+00:00                    None


## Grouping Posts by user

In [13]:
# Group posts by did_id and count posts per user
posts_by_user = posts_table.group_by("did_id").aggregate([
    ("created_at", "count"),  # Total post count
    ("created_at", "min"),    # First post date
    ("created_at", "max")     # Last post date
]).to_pandas()

posts_by_user.columns = ["did_id", "total_posts", "first_post_date", "last_post_date"]
print(f"Unique users with posts: {len(posts_by_user)}")

# Let's see what constitutes "sufficient" posts
post_counts = posts_by_user["total_posts"].value_counts().sort_index()
print("\nPost count frequency:")
for count, freq in post_counts.head(20).items():
    print(f"Users with {count} posts: {freq}")

Unique users with posts: 130113

Post count frequency:
Users with 1 posts: 37137
Users with 2 posts: 16712
Users with 3 posts: 10393
Users with 4 posts: 7173
Users with 5 posts: 5510
Users with 6 posts: 4367
Users with 7 posts: 3503
Users with 8 posts: 2970
Users with 9 posts: 2379
Users with 10 posts: 2128
Users with 11 posts: 1841
Users with 12 posts: 1707
Users with 13 posts: 1449
Users with 14 posts: 1350
Users with 15 posts: 1284
Users with 16 posts: 1079
Users with 17 posts: 988
Users with 18 posts: 994
Users with 19 posts: 819
Users with 20 posts: 869


In [14]:
# Filter out users with 1 or 2 posts
total_users_before = len(posts_by_user)
user_data_filtered = posts_by_user[posts_by_user['total_posts'] > 2]
print(f"Users after filtering: {len(user_data_filtered)} ({len(user_data_filtered)/total_users_before*100:.1f}% remaining)")

Users after filtering: 76264 (58.6% remaining)


## Joining with Profiles
Now let's efficiently add profile information (bluesky joining date, and joined via starter pack) without large joins. 

In [20]:
def add_profile_info_efficiently(user_data_filtered, profiles_path, posts_path):
    """Enhanced version that collects more detailed post information for weekly analysis"""
    print("\nAdding profile information efficiently with enhanced post analysis...")
    
    # Get the list of user IDs we need profile info for
    user_ids_needed = set(user_data_filtered['did_id'].tolist())
    print(f"Need profile info for {len(user_ids_needed)} users")
    
    # Load profiles in chunks and filter for only the users we need
    profiles_chunks = pq.ParquetFile(profiles_path)
    profile_info = {}
    
    for i, batch in enumerate(profiles_chunks.iter_batches(batch_size=50000)):
        print(f"Processing profiles batch {i+1}...")
        batch_df = batch.to_pandas()
        
        # Filter only the users we need
        batch_filtered = batch_df[batch_df['did_id'].isin(user_ids_needed)]
        
        # Store the profile information
        for _, row in batch_filtered.iterrows():
            did_id = row['did_id']
            profile_info[did_id] = {
                'joined_date': row['created_at'],
                'joined_via_starter_pack': row.get('joined_via_starter_pack', False)
            }
        
        # Early exit if we've found all users
        if len(profile_info) >= len(user_ids_needed):
            print(f"Found all {len(user_ids_needed)} users in profile data")
            break
    
    print(f"Retrieved profile info for {len(profile_info)} users")
    
    # Add profile information to our user data
    user_data_filtered['joined_date'] = user_data_filtered['did_id'].map(
        lambda x: profile_info[x]['joined_date'] if x in profile_info else None
    )
    user_data_filtered['joined_via_starter_pack'] = user_data_filtered['did_id'].map(
        lambda x: profile_info[x]['joined_via_starter_pack'] if x in profile_info else None
    )
    
    # Remove any users where we couldn't find profile info
    user_data_complete = user_data_filtered.dropna(subset=['joined_date'])
    print(f"Users with profile info: {len(user_data_complete)}")
    
    # NOW: Collect more detailed post information for weekly analysis
    print("\nCollecting detailed post information for weekly analysis...")
    
    # Convert to datetime for calculations
    user_data_complete['first_post_date'] = pd.to_datetime(user_data_complete['first_post_date'])
    user_data_complete['joined_date'] = pd.to_datetime(user_data_complete['joined_date'])
    
    # Initialize weekly activity trackers
    weekly_activity = {did_id: {
        'first_week_posts': 0,
        'second_week_posts': 0,
        'first_week_active_days': set(),
        'second_week_active_days': set()
    } for did_id in user_data_complete['did_id']}
    
    # Process posts in chunks to count weekly activity
    posts_chunks = pq.ParquetFile(posts_path)
    processed_users = set()
    
    for i, batch in enumerate(posts_chunks.iter_batches(batch_size=50000)):
        print(f"Analyzing posts batch {i+1} for weekly activity...")
        batch_df = batch.to_pandas()
        batch_df['created_at'] = pd.to_datetime(batch_df['created_at'])
        
        # Filter only our users of interest
        batch_filtered = batch_df[batch_df['did_id'].isin(user_data_complete['did_id'])]
        
        for _, row in batch_filtered.iterrows():
            user_id = row['did_id']
            post_date = row['created_at']
            join_date = user_data_complete[user_data_complete['did_id'] == user_id]['joined_date'].iloc[0]
            
            # Calculate days since joining
            days_since_join = (post_date - join_date).days
            
            # Count first week activity (days 0-6)
            if 0 <= days_since_join <= 6:
                weekly_activity[user_id]['first_week_posts'] += 1
                weekly_activity[user_id]['first_week_active_days'].add(days_since_join)
            
            # Count second week activity (days 7-13) - OUR TARGET
            elif 7 <= days_since_join <= 13:
                weekly_activity[user_id]['second_week_posts'] += 1
                weekly_activity[user_id]['second_week_active_days'].add(days_since_join)
        
        # Track progress
        batch_users = set(batch_filtered['did_id'])
        processed_users.update(batch_users)
        print(f"  Processed {len(processed_users)} unique users so far...")
    
    # Add weekly activity features to our dataset
    user_data_complete['first_week_posts'] = user_data_complete['did_id'].map(
        lambda x: weekly_activity[x]['first_week_posts']
    )
    user_data_complete['second_week_posts'] = user_data_complete['did_id'].map(
        lambda x: weekly_activity[x]['second_week_posts']
    )
    user_data_complete['first_week_active_days'] = user_data_complete['did_id'].map(
        lambda x: len(weekly_activity[x]['first_week_active_days'])
    )
    user_data_complete['second_week_active_days'] = user_data_complete['did_id'].map(
        lambda x: len(weekly_activity[x]['second_week_active_days'])
    )
    
    # Create derived features
    user_data_complete['posted_in_first_week'] = user_data_complete['first_week_posts'] > 0
    user_data_complete['active_in_second_week'] = user_data_complete['second_week_posts'] > 0
    user_data_complete['first_week_posting_intensity'] = user_data_complete['first_week_posts'] / user_data_complete['first_week_active_days'].replace(0, 1)
    
    # Calculate days to first post
    user_data_complete['days_to_first_post'] = (user_data_complete['first_post_date'] - user_data_complete['joined_date']).dt.days
    
    # Additional engagement features
    user_data_complete['total_active_days_ratio'] = user_data_complete['first_week_active_days'] / 7.0
    user_data_complete['early_poster'] = user_data_complete['days_to_first_post'] <= 1
    
    print(f"\nFinal dataset with enhanced features: {len(user_data_complete)} users")
    
    # Print summary statistics
    print("\n=== WEEKLY ACTIVITY SUMMARY ===")
    print(f"Users who posted in first week: {user_data_complete['posted_in_first_week'].sum()} ({user_data_complete['posted_in_first_week'].mean()*100:.1f}%)")
    print(f"Users active in second week: {user_data_complete['active_in_second_week'].sum()} ({user_data_complete['active_in_second_week'].mean()*100:.1f}%)")
    print(f"Average first week posts: {user_data_complete['first_week_posts'].mean():.2f}")
    print(f"Average second week posts: {user_data_complete['second_week_posts'].mean():.2f}")
    print(f"Average active days in first week: {user_data_complete['first_week_active_days'].mean():.2f}")
    
    return user_data_complete

In [ ]:
# Add profile information efficiently
user_data_with_profiles = add_profile_info_efficiently(user_data_filtered, posts_path,profiles_path)

print("\nFinal dataset preview:")
print(user_data_with_profiles[['did_id', 'total_posts', 'first_post_date', 'joined_date', 'joined_via_starter_pack']].head())


Adding profile information efficiently with enhanced post analysis...
Need profile info for 76264 users
Processing profiles batch 1...
Processing profiles batch 2...
Processing profiles batch 3...
Processing profiles batch 4...
Processing profiles batch 5...
Processing profiles batch 6...
Processing profiles batch 7...
Processing profiles batch 8...
Processing profiles batch 9...
Processing profiles batch 10...
Processing profiles batch 11...
Processing profiles batch 12...
Processing profiles batch 13...
Processing profiles batch 14...
Processing profiles batch 15...
Processing profiles batch 16...
Processing profiles batch 17...
Processing profiles batch 18...
Processing profiles batch 19...
Processing profiles batch 20...
Processing profiles batch 21...
Processing profiles batch 22...
Processing profiles batch 23...
Processing profiles batch 24...
Processing profiles batch 25...
Processing profiles batch 26...
Processing profiles batch 27...
Processing profiles batch 28...
Processi

In [18]:
# Save the processed data for future use
output_path = "/home/ale/Documents/uni/mp/data/posting_behaviour/processed/user_activity_analysis.parquet"
user_data_with_profiles.to_parquet(output_path, index=False)
print(f"\nData saved to: {output_path}")


Data saved to: /home/ale/Documents/uni/mp/data/posting_behaviour/processed/user_activity_analysis.parquet
